# The `class` keyword

A **class** is a way to group related data and behavior under one concept.

An **attribute** is a piece of information stored inside an object. It represents the object's state at a given moment. 

A **method** is a function that belongs to a class. It defines what the object can do and how its state can be read or changed. 

An **object** is a concrete instance of a class with its own attribute values.

**Why is this important?**
Throughout the lab you will meet classes in most labs. Being familiar with both the concept behind it but also the syntax quirks will give you an advantage. 


## The `drone` class
We will use the `Drone` class below to show you how a class works. We will re-use most of the `update_dynamics` method and modify it to include yaw logic. 


In [74]:
import numpy as np

class Drone:
    def __init__(self, x0, name, yaw0=0.0):
        """
        x0   : initial position as a numpy array, e.g. np.array([0.0, 0.0, 0.0])
        name : string identifier for the drone
        yaw0 : initial yaw angle in radians (default 0.0)
        """
        self.name = name
        self.position = x0.copy()
        self.yaw = yaw0
        self.trajectory = [x0.copy()]  # stores all positions
        self.yaw_history = [yaw0]      # stores yaw at each timestep
        self.timesteps = [0.0]         # stores corresponding time values

    def update_dynamics(self, velocity,  yaw,dt):
        """
        Advance the drone one step forward in time.

        velocity : numpy array with the same shape as x0
        dt       : simulation timestep in seconds
        yaw      : yaw angle in radians for this timestep
        """
        self.position = self.position + velocity * dt
        self.yaw = yaw
        self.trajectory.append(self.position.copy())
        self.yaw_history.append(self.yaw)
        self.timesteps.append(self.timesteps[-1] + dt)

## Understanding `self` and `__init__`

- `self` represents the current object instance.
- Inside a method, writing `self.position` means "the `position` attribute of this specific object."
- Python passes `self` automatically when you call a method with `object.method(...)`.

### The `__init__` method

- `__init__` is the initialization method of a class.
- It runs automatically when a new object is created, for example with `Drone(x0, name)`.
- Use it to define the initial state of the object (attributes such as `name`, `position`, and logs like `trajectory`).

### Important class syntax details

- Every instance method must have `self` as its first parameter.
- Attributes that belong to each object should be stored as `self.attribute_name`.
- Methods can both read and modify object state, which is how behavior is attached to data.

### Mission

Try it out. Create a `Drone`, give it a constant velocity, and simulate a few steps.

In [ ]:
# Your code here
x0 = np.array([0.0, 0.0, 0.0])   # start at the origin
v  = np.array([1.0, 0.5, 0.0])   # constant velocity (m/s)
yaw= 0.0                          # constant yaw (radians)
dt = 0.1                          # timestep (s)

drone = # TODO: create a Drone instance with x0 and a name of your choice

for _ in range(10):
    pass # TODO: update the drone's dynamics for 10 timesteps using the velocity v and a yaw of 0.0

print(f"Drone: {drone.name}")
print(f"Final position : {drone.position}")
print(f"Trajectory ({len(drone.trajectory)} points):")
for t, pos in zip(drone.timesteps, drone.trajectory):
    print(f"  t={t:.1f}s  pos={pos}")

### Mission
We will now create our "simulation" for the Crazyflie drone. We will write a class `CrazyFlieSim` together. We will provide some helper functions and some of the structure. Your tasks will be:
- Pass and store required initialization parameters (`max_speed`, `dt`, `drone`) to the `__init__` method. 
- Initialize the missing class attributes. 
- Define a `simulate` method that:
    - Takes a NumPy array of `velocity_commands`
    - Saturates the velocity command so that the drone never exceeds `max_speed` 
    - Updates the drone's position using the `update_dynamics` method with this timesteps' command and the defined `dt` in the `__init__` method



In [ ]:
import numpy as np
import matplotlib.pyplot as plt


class CrazyFlieSim:
    def __init__(self, max_speed, dt, drone):
        # TODO: Store max_speed, dt, and the drone instance as attributes.

    def simulate(self, velocity_commands, yaw_commands):
        """Simulate a sequence of velocity and yaw commands.

        velocity_commands : iterable of 3D velocity vectors (m/s)
        yaw_commands      : iterable of yaw angles in radians, one per velocity command
        """
        for v_cmd, yaw in zip(velocity_commands, yaw_commands):
            v_cmd = np.asarray(v_cmd, dtype=float)
            speed = np.linalg.norm(v_cmd)

            # TODO: Saturate velocity if command exceeds max speed.

            # TODO: Update drone dynamics with the (possibly saturated) velocity command and yaw.

    def plot_trajectory(self, arrow_scale=0.1):
        """Plot 3D trajectory, start/end points, and yaw arrows.

        arrow_scale : length of each yaw arrow in metres (default 0.1).
        """
        traj = np.array(self.drone.trajectory)
        yaws = np.array(self.drone.yaw_history)

        fig = plt.figure(figsize=(7, 5))
        ax = fig.add_subplot(111, projection="3d")
        ax.plot(traj[:, 0], traj[:, 1], traj[:, 2], marker="o", label=self.drone.name)
        ax.scatter(traj[0, 0], traj[0, 1], traj[0, 2], color="green", s=60, label="start")
        ax.scatter(traj[-1, 0], traj[-1, 1], traj[-1, 2], color="red", s=60, label="end")

        # Draw yaw arrows as a single vectorized quiver call.
        ax.quiver(traj[:, 0], traj[:, 1], traj[:, 2],
                  arrow_scale * np.cos(yaws),
                  arrow_scale * np.sin(yaws),
                  np.zeros_like(yaws),
                  normalize=False, color="orange", arrow_length_ratio=0.3)

        ax.set_xlabel("x [m]")
        ax.set_ylabel("y [m]")
        ax.set_zlabel("z [m]")
        ax.set_title(f"3D Trajectory of {self.drone.name}")
        ax.legend()
        plt.tight_layout()
        plt.show()


# Example usage
x0 = np.array([0.0, 0.0, 0.0])
drone = Drone(x0=x0, name="cf1")
sim = CrazyFlieSim(max_speed=1.2, dt=0.1, drone=drone)
velocity_commands = [
    [1.0, 0.2, 0.0],
    [1.5, 0.3, 0.0],
    [0.4, 1.0, 0.0],
    [0.2, -0.8, 0.0],
    [1.7, 0.0, 0.0],
]
yaw_commands = [0.0, 0.2, 0.8, -0.5, 0.0]

sim.simulate(velocity_commands, yaw_commands)
sim.plot_trajectory(arrow_scale=0.1)

### Mission 
Now use NumPy to define 100 velocity commands so that the simulated drone follows an upwards spiral trajectory. To help you we provide a parametrized form of a spiral where $r$ is the radius of the spiral, $v_z$ is the velocity upwards and $t$ is the time.  
$$ \mathbf{x(t)} = [r * \cos(t), r \sin(t), v_z \cdot t]$$
$$ \mathbf{x'(t)} = [-r * \sin(t), r \cos(t), v_z]$$

Now use that array of velocity commands to simulate a new trajectory. 
drone = Drone(x0=x0, name="cf1")
sim = CrazyFlieSim(max_speed=1.2, dt=0.1, drone=drone)

In [ ]:
drone = Drone(x0=x0, name="cf1")
sim = CrazyFlieSim(max_speed=1.2, dt=0.1, drone=drone)

# Spiral parameters
r = 1.0          # radius [m]
vz = 0.2         # upward velocity [m/s]
f_hz = 0.4       # spiral frequency [Hz]
num_commands = 100
dt = sim.dt      # simulation time step [s]

# Time vector (one command per simulation step) 
t = np.arange(num_commands) * dt

# TODO: compute omega (angular velocity) from f_hz

# TODO: compute velocity_commands as a (100, 3) NumPy array for the spiral

# TODO: compute yaw_commands (Yaw follows the tangent direction of the spiral, alternatively you can experiment with different yaw profiles if you like)

sim.simulate(velocity_commands, yaw_commands)
sim.plot_trajectory()
